In [3]:
# !pip install torch torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv -q
# (On some platforms you may need platform-specific wheels; see https://pytorch-geometric.readthedocs.io/)

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

# --- Repro & device ---
seed = 42
torch.manual_seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Data: Cora with normalized node features ---
dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
data = dataset[0].to(device)

# --- Model: 2-layer GAT (8 heads -> 8*8 hidden, then 1 head to num_classes) ---
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads_1=8, heads_2=1, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads_1, dropout=dropout)
        # concat=True by default on GATConv, so output dim of layer1 is hidden_dim * heads_1
        self.gat2 = GATConv(hidden_dim * heads_1, out_dim, heads=heads_2,
                            concat=False, dropout=dropout)  # concat=False for final logits

    def forward(self, x, edge_index):
        print("x", x.shape)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index)
        return x  # logits

model = GAT(
    in_dim=dataset.num_node_features,
    hidden_dim=8,
    out_dim=dataset.num_classes,
    heads_1=8,
    heads_2=1,
    dropout=0.6
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

def accuracy(logits, y):
    return (logits.argmax(dim=-1) == y).float().mean().item()

best_val_acc = 0.0
best_state = None

for epoch in range(1, 401):
    # --- Train ---
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    # --- Eval ---
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        train_acc = accuracy(logits[data.train_mask], data.y[data.train_mask])
        val_acc = accuracy(logits[data.val_mask], data.y[data.val_mask])
        test_acc = accuracy(logits[data.test_mask], data.y[data.test_mask])

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | loss {loss.item():.4f} | "
              f"train {train_acc:.3f} | val {val_acc:.3f} | test {test_acc:.3f}")

# --- Load best-by-val and report test ---
if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    final_test_acc = accuracy(logits[data.test_mask], data.y[data.test_mask])
print(f"\nBest val acc: {best_val_acc:.3f} | Test acc (at best val): {final_test_acc:.3f}")


x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
Epoch   1 | loss 1.9495 | train 0.250 | val 0.204 | test 0.200
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708, 1433])
x torch.Size([2708,